# CS 3892 / 5892 — Session 25 · Neural-network verification II

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-12-01-nn-verification-tools-and-specs.ipynb)

**Tuesday, December 1, 2026.** Two verifiers on the same 25 queries — **CROWN** (sound, incomplete)
and an exact **MILP** (sound and complete) — and the one cell of their agreement table that must stay
empty; a **buggy** verifier the cross-check catches; what **timeout** means; and an **LLM writing
VNN-LIB**, checked mechanically. Session 24's toy network again (2 → 2 → 2 ReLU). This covers HW5
Parts 3 and 5; HW5 is due Friday Dec 4.

## Setup

Run these two once. The second installs HW5's pinned tools: a minute or two.

In [ ]:
# --- Setup: find the repo (clone on Colab), define helpers
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-12-01-nn-verification-tools-and-specs"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

PYD = ROOT / "sessions" / SESSION / "python"
HD  = ROOT / "homework" / "hw5-nn-verification"
FIG = ROOT / "sessions" / SESSION / "figures"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True, cwd=str(path.parent))
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

from IPython.display import SVG, display
print("ready — helpers: show(path), run(path)")

In [ ]:
# --- The tools, HW5's pinned set: torch 2.11.0 CPU (about 200 MB), auto_LiRPA 0.8.0 at its release
# commit, numpy. A minute or two the first time.
pip = [sys.executable, "-m", "pip", "install", "--quiet"]
subprocess.run(pip + ["-r", str(HD / "requirements.txt")], check=True)
subprocess.run(pip + ["--no-deps", "--ignore-requires-python", "-r", str(HD / "requirements-lirpa.txt")], check=True)
import torch, auto_LiRPA
print("torch", torch.__version__, "| auto_LiRPA", auto_LiRPA.__version__)

## 1. An exact verifier, and two verifiers on the same queries

`milp.py` encodes each ReLU exactly with one binary variable. CROWN against it, on 25 queries.

In [ ]:
show(PYD / "toy.py"); show(PYD / "queries.py"); show(PYD / "milp.py")
out = run(PYD / "01_two_verifiers.py")
assert "no unsound disagreement" in out

## 2. What the cross-check is for: one plausible bug

CROWN with the chord's intercept dropped. The MILP catches it — by verdict, and more often by bound.

In [ ]:
out = run(PYD / "02_buggy_verifier.py")
assert "4 of 25" in out

## 3. Timeout and unknown prove nothing

One query, three budgets.

In [ ]:
out = run(PYD / "03_timeout.py")
assert "not about the network" in out

## 4. An LLM writes VNN-LIB; a checker decides

The prompt and the reply are in `llm/`, verbatim (Claude Haiku 4.5, no tools, one attempt).

In [ ]:
print((PYD.parent / "llm" / "prompt.txt").read_text())
out = run(PYD / "04_llm_vnnlib.py")
assert "all three files match" in out

## Try it yourself

- In `02_buggy_verifier.py`, invent a different bug (say, the lower slope 1 when it should be 0).
  Does the cross-check catch it? On how many queries?
- Write Q4 in English yourself — "class 1 wins everywhere in the box 0.1 ≤ X_0 ≤ 0.3,
  0.8 ≤ X_1 ≤ 1" — then its VNN-LIB, and check it with `hw5lib.parse_vnnlib` and `milp.solve`.
- Ask an assistant for Q4. Does its file match yours?